Initialize clients


In [ ]:
import os
import chromadb
from google import genai
from google.genai import types

# Paste your API key string here:
client = genai.Client(api_key="") #has been left out as .env doc hasn't been set up just yet

chroma_client = chromadb.Client()  # Use chromadb.PersistentClient(path="./rag_db") for persistence
collection = chroma_client.get_or_create_collection(name="knowledge_base")

EMBEDDING_MODEL = "models/gemini-embedding-2"

Chunking

In [24]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Simple sliding-window chunker based on characters/words."""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

Doc ingestion & embedding

In [51]:
import time
from google.genai.errors import ClientError

def ingest_documents(docs: list[dict], batch_size: int = 8, delay_between_batches: float = 2.0):
    """
    Ingests document chunks with pacing and exponential backoff 
    to respect free-tier rate limits.
    """
    chunk_ids = []
    chunk_texts = []
    chunk_embeddings = []
    metadatas = []

    all_chunks = []
    all_metadatas = []

    # 1. Flatten all chunks from all pages
    for doc in docs:
        chunks = chunk_text(doc["text"])
        for chunk in chunks:
            all_chunks.append(chunk)
            all_metadatas.append({"source": doc.get("title", "Unknown")})

    if not all_chunks:
        print("No text extracted from PDF.")
        return

    print(f"Total chunks to embed: {len(all_chunks)}. Ingesting in batches of {batch_size}...")

    # 2. Batch process with rate-limit retries and pacing
    global_idx = 0
    total_batches = (len(all_chunks) + batch_size - 1) // batch_size

    for batch_num in range(total_batches):
        start = batch_num * batch_size
        batch_texts = all_chunks[start : start + batch_size]
        batch_meta = all_metadatas[start : start + batch_size]

        # Retry logic with exponential backoff for 429 errors
        max_retries = 5
        wait_seconds = 10.0

        for attempt in range(max_retries):
            try:
                # Process each chunk in the batch individually
                for text, meta in zip(batch_texts, batch_meta):
                    response = client.models.embed_content(
                        model=EMBEDDING_MODEL,
                        contents=text, # Pass a single string
                        config=types.EmbedContentConfig(
                            task_type="RETRIEVAL_DOCUMENT"
                        )
                    )
                    
                    chunk_ids.append(f"doc_{global_idx}")
                    chunk_texts.append(text)
                    # Extract the single embedding value
                    chunk_embeddings.append(response.embeddings[0].values)
                    metadatas.append(meta)
                    global_idx += 1

                print(f"Successfully processed batch {batch_num + 1}/{total_batches}")
                break  # Batch succeeded

            except ClientError as e:
                if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                    if attempt < max_retries - 1:
                        print(f"Hit 429 rate limit on batch {batch_num + 1}. Waiting {wait_seconds}s before retry...")
                        time.sleep(wait_seconds)
                        wait_seconds *= 2 
                    else:
                        raise e
                else:
                    raise e
        # Polite delay between successful calls to stay under RPM quotas
        time.sleep(delay_between_batches)

    # 3. Add to ChromaDB
    collection.add(
        ids=chunk_ids,
        documents=chunk_texts,
        embeddings=chunk_embeddings,
        metadatas=metadatas
    )
    print(f"Done! Indexed {len(chunk_texts)} chunks successfully into ChromaDB.")

Retrieval

In [40]:
def retrieve(query: str, top_k: int = 10) -> list[dict]:
    """Generates query embedding using the same model identifier."""
    query_resp = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    query_vector = query_resp.embeddings[0].values

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )

    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({"content": doc, "metadata": meta})
    return retrieved

Generation Step (RAG Synthesis)

In [33]:
def generate_rag_response(query: str, retrieved_passages: list[dict]) -> str:
    """Formats retrieved context into the prompt and queries Gemini."""
    context_blocks = "\n\n".join(
        [f"[Source: {p['metadata']['source']}]\n{p['content']}" for p in retrieved_passages]
    )

    system_instruction = (
        "You are an accurate factual assistant. Answer user questions strictly using "
        "the provided Context. If the context does not contain enough information, state that "
        "you do not know rather than speculating. Always cite sources."
    )

    prompt = f"""Context:
{context_blocks}

User Question: {query}
"""

    response = client.models.generate_content(
        model='models/gemini-3.5-flash',  # <--- Updated model name
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2
        )
    )
    return response.text

example

In [53]:
# Reset the collection completely before re-ingesting
try:
    chroma_client.delete_collection("knowledge_base")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="knowledge_base")

if __name__ == "__main__":
    pdf_path = "sample_report.pdf"  # Path to your PDF file

    # 1. Parse PDF pages into document records
    pdf_docs = load_pdf("BArtificialIntelligenceTER.pdf")

    # 2. Ingest into ChromaDB using your existing function
    ingest_documents(pdf_docs)

    # 3. Retrieve and query
    user_query = "Minimum EC requirement for positive reccomendation on continuation of studies?" 
    passages = retrieve(user_query, top_k=10)
    answer = generate_rag_response(user_query, passages)

    print("\n--- Output ---")
    print(answer)

Total chunks to embed: 41. Ingesting in batches of 8...
Successfully processed batch 1/6
Successfully processed batch 2/6
Successfully processed batch 3/6
Successfully processed batch 4/6
Successfully processed batch 5/6
Successfully processed batch 6/6
Done! Indexed 41 chunks successfully into ChromaDB.

--- Output ---
Based on the provided context, to obtain a positive recommendation on the continuation of studies, a student must have obtained **at least 42 EC** of first-year courses of the programme by the end of the first year of enrolment [Source: BArtificialIntelligenceTER.pdf - Page 22].


PDF


In [42]:
from pypdf import PdfReader

def load_pdf(file_path: str) -> list[dict]:
    """
    Reads a PDF file and returns a list of dictionaries with text and metadata.
    Preserves page numbers to enable page-level citations.
    """
    reader = PdfReader(file_path)
    documents = []
    
    file_name = file_path.split("/")[-1]

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text and text.strip():
            documents.append({
                "title": f"{file_name} - Page {page_number}",
                "text": f"[Document: {file_name} | Page {page_number}]\n{text}"
            })
            
    return documents

In [36]:
for m in client.models.list():
    if "embedContent" in (m.supported_actions or []):
        print(m.name)

for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview

In [39]:
from pypdf import PdfReader

reader = PdfReader("BArtificialIntelligenceTER.pdf")
print("=== PAGE 22 TEXT ===")
print(reader.pages[21].extract_text())  # index 21 = Page 22

=== PAGE 22 TEXT ===
2026-2027 Teaching and Examination Regulations Bachelor 22/31
14. Binding recommendation on continuation of studies (BSA)
Article 14.1 Binding (negative) recommendation
1. In order to obtain a positive recommendation on continuation of studies, the
student must have obtained:
a. at least 42 EC of first-year courses of the programme by the end of the
first year of enrolment;
Advice OLC (7.13
f)
2. Students who receive a binding negative recommendation on continuation of
studies cannot enrol in the following bachelor’s programme(s) offered by the
Faculty during the subsequent three academic years:
- Artificial Intelligence
- Computer Science
Advice OLC (7.13
f)
3. Not applicable. Advice OLC (7.13
f)
15. Evaluation and transitional provisions
Article 15.1 Evaluation of the programme
The education provided in this programme is evaluated in accordance with the
document ‘Kwaliteitszorg Onderwijs BETA’.
Approval OLC
(7.13 a1)
Article 15.2 Transitional provisions
By way of